# Install & Import

In [11]:
!pip install -q sentence-transformers faiss-cpu rank_bm25 transformers #already added in the dependency installation code

In [4]:
import pandas as pd
import numpy as np
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder


# load dataset

In [5]:
dirname= "/kaggle/input/competitions/llm-agentic-legal-information-retrieval"

In [ ]:
train = pd.read_csv(dirname+"/train.csv")
# val = pd.read_csv(dirname+"/val.csv")
# test = pd.read_csv(dirname+"/test.csv")
laws_de = pd.read_csv(dirname+"/laws_de.csv")
court_considerations = pd.read_csv(dirname+"/court_considerations.csv")

In [7]:
print(f'''
"train":{train.shape},
"\nlaws_de:"{laws_de.shape},
"\ncourt_considerations:"{court_considerations.shape}''')


"train":(1139, 3),
"
laws_de:"(175933, 3),
"
court_considerations:"(2476315, 2)


In [9]:
train.head()

,query_id,query,gold_citations
0,train_0001,Die A AG betreibt seit den 1970er-Jahren auf d...,Art. 10a Abs. 1 USG;Art. 2 Abs. 1 UVPV;Art. 10...
1,train_0002,Die Alpha Consulting AG kann nun ihr Grundstüc...,Art. 975 ZGB;Art. 974 Abs. 2 ZGB;Art. 973 ZGB;...
2,train_0003,Das Kompetenzzentrum Völkerstrafrecht bei der ...,Art. 264m StGB
3,train_0004,Die Linzer Stadtbahn AG ('LiSA') ist die priva...,Art. 176 Abs. 1 IPRG;Art. 186 Abs. 1 IPRG;Art....
4,train_0005,Die Stadt Winterthur beschloss am 10. Februar ...,Art. 93 Abs. 1 BGG;Art. 89 Abs. 1 BGG;Art. 89 ...


In [10]:
laws_de.head()

,citation,text,title
0,Art. 1 112,Die Einwohnergemeinde Bern tritt der Schweizer...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
1,Art. 2 112,Die Einwohnergemeinde Bern wird ferner der Sch...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
2,Art. 3 Abs. 1 112,1 Falls die Schweizerische Eidgenossenschaft z...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
3,Art. 3 Abs. 2 112,2 Durch Anlage des neuen Verwaltungsgebäudes a...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
4,Art. 4 Abs. 1 112,1 Die Einwohnergemeinde Bern übernimmt im fern...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...


In [11]:
court_considerations.head()

,citation,text
0,BGE 139 I 2 E. 1.12.2011,betr. Verweigerung der Beiladung seien gutzuhe...
1,BGE 139 I 2 E. 2,Eventualiter sei die Rückweisung an die Vorins...
2,BGE 139 I 2 E. 5.1,"In der Sache ist vorweg zu prüfen, ob der Ents..."
3,BGE 139 I 2 E. 5.2,Art. 34 Abs. 1 BV gewährleistet in allgemeiner...
4,BGE 139 I 2 E. 5.3,Im vorliegenden Fall geht es nicht um die Gült...


laws_de is like law book

court_considerations is like case studies

| laws_de          | court_considerations |
| ---------------- | -------------------- |
| Rules            | Interpretations      |
| Short text       | Long text            |
| Structured       | Complex              |
| Easier retrieval | Harder retrieval     |


# Prepare Corpus

In [12]:
laws_de.columns

Index(['citation', 'text', 'title'], dtype='object')

In [5]:
#prepare corpus
def prepare(df):
    df["title"]=df.get("title","").fillna("")
    df["text"]=df["text"].fillna("")
    df["full_text"]=df["title"]*3+" "+df["text"]
    return df

laws_corpus=prepare(laws_de)

laws_docs = laws_corpus["full_text"].tolist()
laws_ids = laws_corpus["citation"].tolist()

court_docs=court_considerations["text"].tolist()
court_ids=court_considerations["citation"].tolist()

id2text ={**dict(zip(laws_docs,laws_ids)), **dict(zip(court_docs,court_ids))} # making big lookup table

SyntaxError: incomplete input (1047038190.py, line 8)

# Model Initialization

In [27]:
# cross Lingual
from sentence_transformers import SentenceTransformer, CrossEncoder
import torch

model_path= "/kaggle/input/models/yethukmutt/bge-m3/transformers/m3/1/bge-m3"
reranker_path="/kaggle/input/models/andreasbis/baai-bge-reranker-v2-m3/transformers/default/1"

embed_model =SentenceTransformer(model_path,model_kwargs={"torch_dtype": "float16"}) # best choice
reranker = CrossEncoder(reranker_path, model_kwargs={"torch_dtype": "float16"})

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

# Translation Batch Precompute

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

tokenizer=MarianTokenizer.from_pretrained(translate_path)
translator=MarianMTModel.from_pretrained(translate_path).to("cpu")

In [ ]:
#Translation;Batch Precompute
def translate(query):
    inputs=tokenizer(query, return_tensors='pt',trancation=True, padding=True)
    outputs=translator(**inputs,max_length=256)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

all_translated={}

for q in tqdm(test['query'],desc="Translating"):
    all_translated[q]=translate(q)

# FAISS index

In [2]:
!pip -q install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 40.7 MB/s eta 0:00:00:00:0100:01


## Law (full)

In [4]:
import faiss
print(faiss.get_num_gpus())

2


In [ ]:
#faiss index; law (full)
import faiss
laws_embed=embed_model.encode(
    laws_docs,
    batch_size=8,
    normalize_embeddings=True,
    show_progress_bar= True,
).astype("float32")

laws_index=faiss.IndexFlatIP(laws_embed.shape[1])
laws_index.add(laws_embed)

## Court Quantize

In [ ]:
court_embed=embed_model.encode(
    court_docs,
    batch_size=8,
    normalize_embedding=True,
    show_progess_bar=True
).astype("float32")

court_dim=court_embed.shape[1]

court_index=faiss.IndexScalarQuantizer(
    court_dim,
    faiss.ScalarQuantization.QT_8bit
)

court_index.train(court_embed)
court_index.add(court_embed)

# BM25

In [ ]:
def tokenize(text):
    return text.lower().replace("."," ").split()

bm25=BM25Okapi([token(t) for t in laws_corpus["full_text"]])

# Intent detection

In [ ]:
def detect_intent(query):
    q = query.lower()

    l = sum(k in q for k in ["article", "art.", "section", "law"])
    c = sum(k in q for k in ["court", "case", "decision", "judgment"])

    if l > c:
        return "law"
    elif c > l:
        return "court"
    return "mixed"

In [ ]:
def vector_search(q, intent):
    emb = embed_model.encode([q], normalize_embeddings=True)

    if intent == "law":
        lk, ck = 40, 10
    elif intent == "court":
        lk, ck = 10, 40
    else:
        lk, ck = 25, 25

    ls, li = law_index.search(emb, lk)
    cs, ci = court_index.search(emb, ck)

    law_res = [(law_ids[i], law_docs[i], s) for i, s in zip(li[0], ls[0])]
    court_res = [(court_ids[i], court_docs[i], s) for i, s in zip(ci[0], cs[0])]

    return law_res + court_res

In [ ]:
def bm25_search(q, k=30):
    scores = bm25.get_scores(tokenize(q))
    idx = np.argsort(scores)[::-1][:k]
    return [(law_ids[i], law_docs[i], scores[i]) for i in idx]

In [ ]:
def dedup(results):
    best = {}
    for doc_id, text, score in results:
        if doc_id not in best or best[doc_id] < score:
            best[doc_id] = score
    return [(k, "", v) for k, v in best.items()]

In [ ]:
def get_weights(intent):
    if intent == "law":
        return {"vector": 1.0, "bm25": 1.2}
    elif intent == "court":
        return {"vector": 1.2, "bm25": 0.5}
    return {"vector": 1.0, "bm25": 1.0}


def weighted_rrf(res, weights, k=60):
    scores = {}
    for key, results in res.items():
        w = weights.get(key, 1.0)
        for rank, (doc_id, _, _) in enumerate(results):
            scores[doc_id] = scores.get(doc_id, 0) + w*(1/(k+rank+1))
    return scores

In [ ]:
def pattern_boost(scores, q):
    if re.search(r"art\.?\s?\d+", q.lower()):
        return {k: v*1.2 if "Art." in k else v for k,v in scores.items()}
    if "bge" in q.lower():
        return {k: v*1.2 if "BGE" in k else v for k,v in scores.items()}
    return scores

In [ ]:
def dynamic_top_k(scores):
    return min(20, max(8, int(len(scores)*0.3)))

In [ ]:
def rerank(query, docs):
    pairs = [[query, f"{doc_id} {text}"] for doc_id, text in docs]

    scores = []
    for i in range(0, len(pairs), 32):
        scores.extend(reranker.compute_score(pairs[i:i+32], normalize=True))

    ranked = sorted(zip(scores, docs), reverse=True)

    k = dynamic_top_k([s for s,_ in ranked])
    return [doc_id for _, (doc_id, _) in ranked[:k]]

In [ ]:
def pipeline(query):
    
    intent = detect_intent(query)

    q_en = query
    q_de = all_translated.get(query, query)

    vec_en = vector_search(q_en, intent)
    vec_de = vector_search(q_de, intent)

    vec = dedup(vec_en + vec_de)

    bm = dedup(bm25_search(q_de))

    fused = weighted_rrf({
        "vector": vec,
        "bm25": bm
    }, get_weights(intent))

    fused = pattern_boost(fused, query)

    top_ids = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:60]

    docs = [(doc_id, id2text[doc_id]) for doc_id,_ in top_ids]

    final = rerank(query, docs)

    return ";".join(final)

In [ ]:
preds = [pipeline(q) for q in tqdm(test["query"])]

sub = pd.DataFrame({
    "query_id": test["query_id"],
    "predicted_citations": preds
})

sub.to_csv("submission.csv", index=False)

In [31]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
